# TEFNET24 Hierarchical Analysis

In [1]:
# Import Dependencies
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import json

# Ignore warnings to have clean cell outputs
import warnings
warnings.filterwarnings("ignore")

### *Create results directory if it doesn't exist*

In [2]:
# Get the current working directory (where the notebook is running)
base_dir = Path.cwd()

project_root = base_dir.parent.parent.parent

# Define the results directory
results_dir = project_root / "results" / "TEFNET24_400G_CAGR_0.2" / "NoBypass"

# Create it if it doesn't exist
results_dir.mkdir(parents = True, 
                  exist_ok = True)

print(f"Results will be saved in: {results_dir}")

Results will be saved in: /media/matin/DataDisk/University/Master/Project/Codes/SixGman_Techno/results/TEFNET24_400G_CAGR_0.2/NoBypass


In [3]:
sys.path.append(os.path.abspath(project_root / 'src'))
from sixdman.core.network import Network
from sixdman.core.band import Band, OpticalParameters
from sixdman.core.planning import PlanningTool
from sixdman.core.post_process import analyse_result

### *1. Create Network Instance*

In [4]:
# Initialize network
network = Network(topology_name = 'TEFNET24')

# Load topology from .mat file
network.load_topology(filepath = project_root / 'data/TEFNET24_merged_adjacency.npz', matrixName ='net_topology')

In [5]:
num_nodes = network.graph.number_of_nodes()
print(f"Number of nodes in the network: {num_nodes}")

Number of nodes in the network: 700


### 2. *Define hierarchy node levels*

In [6]:
HL4_Standalone = np.load(project_root / "data/TEFNET24_hierarchical_levels.npz")['HL4_StandAlone'].tolist()
HL4_colocated = np.load(project_root / "data/TEFNET24_hierarchical_levels.npz")['HL4_colocated'].tolist()

HL3_Standalone = np.load(project_root / "data/TEFNET24_hierarchical_levels.npz")['HL3_StandAlone'].tolist()
HL3_colocated = np.load(project_root / "data/TEFNET24_hierarchical_levels.npz")['HL3_colocated'].tolist()

HL2_Standalone = np.load(project_root / "data/TEFNET24_hierarchical_levels.npz")['HL2_StandAlone'].tolist() \
    + np.load(project_root / "data/TEFNET24_hierarchical_levels.npz")['HL1_StandAlone'].tolist()

HL23_Standalone = HL2_Standalone + HL3_Standalone

arr = np.arange(network.adjacency_matrix.shape[0]) 
exclude_list = HL2_Standalone + HL3_Standalone + HL4_Standalone
Optical_Nodes = arr[~np.isin(arr, exclude_list)]

# Set hierarchical levels
hl_dict = network.define_hierarchy(
    HL2_standalone = HL2_Standalone,
    HL3_standalone = HL3_Standalone,
    HL4_standalone = HL4_Standalone, 
    HL4_colocated = HL4_colocated, 
    HL3_colocated = HL3_colocated
)

### *2.1 Load all nodes in the network*

In [7]:
merged_nodes = np.load(project_root / "data/TEFNET24_merged_nodes.npz")['merged_nodes'].tolist()

### *2.2 Load all src_dest pairs*

In [8]:
src_dest_pairs = pd.read_csv(project_root / "data/TEFNET24_src_dest_merged.csv")

In [9]:
src_dest_pairs.head()

,Source,Destination
0,0,"[11, 12]"
1,1,"[11, 12]"
2,2,"[11, 12]"
3,3,"[11, 12]"
4,4,"[11, 12]"


### *2.3 Load number of spans of all the links*

In [10]:
nSpan_merged = np.load(project_root / "data/TEFNET24_nSpan_merged.npz")['nSpan_merged']
nSpan_merged = nSpan_merged.flatten()

### *3. Define Transmission Bands*

In [11]:
# Define C-band parameters
c_band_params = OpticalParameters(Rs_mat = 64e9)

# Create C-band instance
c_band = Band(
    name='C',
    start_freq = 190.65, # THz
    end_freq = 196.675, # THz
    opt_params = c_band_params,
    network_instance = network,
    channel_spacing = 0.075 # THz
    )

# Define L-band parameters
l_band_params = OpticalParameters()

# Create L-band instance
l_band = Band(
    name='L',
    start_freq = 184.525, # THz
    end_freq = 190.565, # THz
    opt_params = l_band_params,
    network_instance = network,
    channel_spacing = 0.075 # THz
)

In [12]:
# define C-band and L-band frequency slots
spectrum_C = c_band.calc_spectrum()
spectrum_L = l_band.calc_spectrum()

# concatenate C-band and KL-band to a sigle frequency spectrum
spectrum = np.concatenate((spectrum_C, spectrum_L))

f_c_axis = spectrum * 1e12  # Convert to Hz
Pch_dBm = np.arange(-6, -0.9, 0.1)  # Channel power in dBm
num_Ch_mat = np.arange(1, len(spectrum) - 1)  # Channel indices

# define total number of frequency slots
num_fslots = len(num_Ch_mat)


C_band_num_channels = int(4800 / 75)
supC_band_num_channels = int(1200 / 75)
L_band_num_channels = int(6000 / 75)

### *4. Create Planning Tool and Optimize Network*

In [13]:
# Initialize planning tool
planner = PlanningTool(
    network_instance = network,
    bands = [c_band, l_band], 
    optical_parameters = c_band_params,
    grid_center = f_c_axis, 
    period_time = 10)

all_node_degree = network.get_node_degrees(np.arange(network.adjacency_matrix.shape[0]))[:, 1]

#### *4.1 Define number of hierarchical levels*

In [14]:
minimum_hierarchy_level = 4
processing_level_list = [4, 3]
splitter = []

### *5. Simulate network level by level*

In [15]:
for i in range(len(processing_level_list)):

    hierarchy_level = processing_level_list[i]

    print(f"Processing hierarchy level: {hierarchy_level}")

    match hierarchy_level:

        case 3:
            HL_Standalone = HL3_Standalone
            HL_colocated = HL3_colocated
            HL_lower_Standalone = HL4_Standalone
            HL_up_Standalone = HL2_Standalone
            HL_all = np.concatenate((HL3_Standalone, HL3_colocated))
            capacity_updt_index = 1
            prev_hierarchy_level = 4
        case 4:
            HL_Standalone = HL4_Standalone
            HL_colocated = HL4_colocated
            HL_lower_Standalone = []
            HL_up_Standalone = HL23_Standalone
            HL_all = np.concatenate((HL4_Standalone, HL4_colocated))
            capacity_updt_index = 2
            prev_hierarchy_level = None

    sub_graph_HL, subnetMatrix_HL = network.compute_hierarchy_subgraph(hierarchy_level, minimum_hierarchy_level)
    splitter.append(len(list(sub_graph_HL.edges())))

    file_name = results_dir / f"{network.topology_name}_HL{hierarchy_level}_K_path_attributes.csv"

    if file_name.exists():

        print(f"Loading K-path attributes of HL{hierarchy_level} ...")

        K_path_attributes_df = pd.read_csv(file_name)
        K_path_attributes_df['links'] = K_path_attributes_df['links'].map(json.loads)
        K_path_attributes_df['nodes'] = K_path_attributes_df['nodes'].map(json.loads)
    else:

        print(f"Calculating K-path attributes of HL{hierarchy_level} ...")

        k_paths = 20
        source_not_found = HL_Standalone.copy()
        while len(source_not_found) != 0:

            # define a list to store path attributes for this iteration
            K_path_attributes = []

            # iterate through each "source_not_found" node only
            for src in HL_Standalone:
                # Calculate candidate destinations from src_dest_pairs dataset
                candidate_dests = json.loads(src_dest_pairs[src_dest_pairs['Source'] == src]['Destination'].values[0])
                for dest in candidate_dests:
                    K_path_attributes = network.compute_k_shortest_paths(subnetMatrix_HL, K_path_attributes, source = src, target = dest, k = k_paths)

            # Convert K_path_attributes list to a temporary DataFrame
            K_path_attributes_df = pd.DataFrame(K_path_attributes)

            # Optionally save to CSV (if you want to track progress)
            K_path_attributes_df.to_csv(file_name, index=False)

            # Sort by num_hops and distance
            K_path_attributes_df_sorted = K_path_attributes_df.groupby(['src_node'], group_keys=False).apply(
                lambda x: x.sort_values(['num_hops', 'distance'])
            )

            # find disjoint pairs from the sorted full dataset
            pairs_disjoint = network.land_pair_finder(HL_Standalone, K_path_attributes_df_sorted, num_pairs=1)

            # Update source_not_found to exclude newly matched source nodes
            source_not_found = np.setdiff1d(HL_Standalone, pairs_disjoint['src_node'].unique())

            print('Remaining src nodes: ', source_not_found)

            k_paths += 20

    file_name = results_dir / f"{network.topology_name}_HL{hierarchy_level}_K_path_attributes_colocated.csv"

    if file_name.exists():

        print(f"Loading K-path attributes of HL{hierarchy_level} colocated...")

        K_path_attributes_colocated_df = pd.read_csv(file_name)
        K_path_attributes_colocated_df['links'] = K_path_attributes_colocated_df['links'].map(json.loads)
        K_path_attributes_colocated_df['nodes'] = K_path_attributes_colocated_df['nodes'].map(json.loads)
    else:

        print(f"Calculating K-path attributes of HL{hierarchy_level} colocated...")

        k_paths = 20

        # define a list to store path attributes
        K_path_attributes_colocated = []

        # iterate through each standalone HL4 node
        for src in HL_colocated:

            candidate_dests = json.loads(src_dest_pairs[src_dest_pairs['Source'] == src]['Destination'].values[0])
            if src in HL3_Standalone:
                candidate_dests = np.unique(candidate_dests + HL3_Standalone)
            elif src in HL2_Standalone:
                candidate_dests = np.unique(candidate_dests + HL2_Standalone)

            for dest in candidate_dests:
                if src != dest:
                    K_path_attributes_colocated = network.compute_k_shortest_paths(subnetMatrix_HL, K_path_attributes_colocated, source = src, target = dest, k = k_paths)

        # Convert K_path_attributes list to dataframe
        K_path_attributes_colocated_df = pd.DataFrame(K_path_attributes_colocated)

        # save dataframe to csv file
        K_path_attributes_colocated_df.to_csv(file_name, index = False)


    # sort dataframes based on num_hops and distance (in order)
    K_path_attributes_df_sorted = K_path_attributes_df.groupby(['src_node'], group_keys = False).apply(lambda x: x.sort_values(['num_hops', 'distance']))
    K_path_attributes_colocated_df_sorted = K_path_attributes_colocated_df.groupby(['src_node', 'dest_node'], group_keys = False).apply(lambda x: x.sort_values(['num_hops', 'distance']))

    # find disjoint pairs for standalone nodes
    pairs_disjoint = network.land_pair_finder(HL_Standalone, K_path_attributes_df_sorted, num_pairs = 1)

    print(f"process link GSMR of HL{hierarchy_level} ...")

    GSNR_opt_link, _, _, power_opt_link = c_band.process_link_gsnr(f_c_axis = f_c_axis,
                                                                    Pch_dBm = Pch_dBm,
                                                                    num_Ch_mat = num_Ch_mat,
                                                                    spectrum_C = spectrum_C,
                                                                    Nspan_array = nSpan_merged,
                                                                    hierarchy_level = hierarchy_level,
                                                                    minimum_hierarchy_level = minimum_hierarchy_level,
                                                                    result_directory = results_dir)

    planner.initialize_planner(num_fslots = num_fslots, 
                               hierarchy_level = hierarchy_level,
                               minimum_hierarchy_level = minimum_hierarchy_level,
                               SR = 64 * 1e9, # Symbol rate of 75G channels (in Hz)
                               Max_bit_rate_BVT = np.array([400, 320, 260, 200, 120, 64]), # 400 Gbps BVTs
                               Ref_license_capacity = np.array([100, 80, 65, 50, 30, 16]), # reference license capacities corresponding to BVTs
                               FP_max_num = 20, 
                               band_sepration_idx = [C_band_num_channels, 
                                                     C_band_num_channels + supC_band_num_channels])


    if hierarchy_level == minimum_hierarchy_level:

        # generate port capacity for HL4 nodes uisng Monte Carlo simulation
        planner.generate_initial_traffic_profile(num_nodes = network.adjacency_matrix.shape[0],
                                                 optical_nodes = Optical_Nodes, 
                                                 monteCarlo_steps = 100,
                                                 min_rate = 20, # Gbps
                                                 max_rate = 200, # Gbps
                                                 seed = 5, 
                                                 result_directory = results_dir)

        # Traffic growth simulation over 10 years
        planner.simulate_traffic_annual(lowest_hierarchy_dict = {'standalone': HL4_Standalone, 'colocated': HL4_colocated},
                                        CAGR = 0.2,
                                        result_directory = results_dir)

    print(f"running planner for HL{hierarchy_level} ...")

    # run the planner for the current hierarchy level
    planner.run_planner(hierarchy_level = hierarchy_level,
                        prev_hierarchy_level = prev_hierarchy_level,
                        pairs_disjoint = pairs_disjoint,
                        kpair_standalone = 1,
                        kpair_colocated = 1,
                        candidate_paths_standalone_df = K_path_attributes_df,
                        candidate_paths_colocated_df = K_path_attributes_colocated_df_sorted,
                        P_opt_links = power_opt_link,
                        GSNR_opt_link = GSNR_opt_link,
                        Nspan_array = np.ones(len(network.all_links)),
                        all_node_degree = all_node_degree,
                        minimum_level = minimum_hierarchy_level,
                        node_cap_update_idx = capacity_updt_index, 
                        result_directory = results_dir)

Processing hierarchy level: 4
Loading K-path attributes of HL4 ...


Loading K-path attributes of HL4 colocated...
process link GSMR of HL4 ...
Loading precomputed link GSNR analysis
Loading precomputed HL_capacity_final ...
Loading precomputed Traffic Matrix ...
running planner for HL4 ...
Processing Year:  1
Processing standalone node 0
Processing standalone node 1
Processing standalone node 2
Processing standalone node 3
Processing standalone node 4
Processing standalone node 5
Processing standalone node 6
Processing standalone node 7
Processing standalone node 8
Processing standalone node 9
Processing standalone node 10
Processing standalone node 13
Processing standalone node 14
Processing standalone node 15
Processing standalone node 16
Processing standalone node 17
Processing standalone node 18
Processing standalone node 19
Processing standalone node 22
Processing standalone node 23
Processing standalone node 24
Processing standalone node 25
Processing standalone node 26
Processing standalone node 27
Processing standalone node 28
Processing standa

In [16]:
analysing = analyse_result(network, 10, processing_level_list, results_dir, save_directory = results_dir)
analysing._load_data()
year = np.arange(1, 11)

In [17]:
cost_df_C_IRU_0v5 = analysing.calc_cost(save_flag = 1, 
                    save_suffix = "_NoBypass_C_IRU_0v5", 
                    C_IRU = 0.5,)

cost_df_C_IRU_0v01 = analysing.calc_cost(save_flag = 1, 
                    save_suffix = "_NoBypass_C_IRU_0v01", 
                    C_IRU = 0.01,)

cost_df_C_IRU_5 = analysing.calc_cost(save_flag = 1, 
                    save_suffix = "_NoBypass_C_IRU_5", 
                    C_IRU = 5,)

In [18]:
cost_df_C_IRU_0v01

,OPEX,Capex_RoB,Capex_MCS,Capex_100GL,CAPEX
0,866.420,3391.500000,168.770000,5664.389000,9224.659000
1,866.420,17.236800,8.832600,407.661354,433.730754
2,866.420,13.851000,9.487301,429.208432,452.546733
3,866.420,6.482268,12.657602,656.329260,675.469130
4,866.420,8.302289,12.786675,1425.092498,1446.181463
5,869.378,13.695431,16.929813,776.433170,807.058414
6,876.514,7.616809,19.448635,1761.284093,1788.349537
7,876.554,17.490086,67.401952,2121.604144,2206.496181
8,882.648,30.869374,105.136674,3131.596254,3267.602302
9,981.568,170.687379,33.070613,2419.251139,2623.009131


In [19]:
node_capacity_profile_array_noBypass = np.load(results_dir / "TEFNET24_HL3_node_capacity_profile_array.npz")['node_capacity_profile_array']

In [20]:
node_capacity_profile_array_noBypass_HL4 = node_capacity_profile_array_noBypass[:, :, 3].cumsum(axis = 0)
node_capacity_profile_array_noBypass_HL3 = node_capacity_profile_array_noBypass[:, :, 2].cumsum(axis = 0)
node_capacity_profile_array_noBypass_HL2 = node_capacity_profile_array_noBypass[:, :, 1].cumsum(axis = 0)

ref_routers_cap = np.array([400, 800, 1600, 3200, 6400, 12800, 25600])
ref_routers_cost_metric = np.array([12, 24, 48, 96, 192, 384, 768]) # cost units in the comprehensive paper
ref_routers_cost_metric = ref_routers_cost_metric * 2000 / 15000 # cost unit 0f paper is 2000, our work is 15000
ref_routers_power_consumption = np.array([2000, 2500, 3100, 3900, 4900, 6100, 7600]) # in Watts
ref_year = 5 # year five is the reference for selection of router

In [21]:
# NoBypass Scenario
HL4_SA_num_routers_noBypass = []
HL4_SA_selected_router_noBypass = []

HL4_co_num_routers_noBypass = []
HL4_co_selected_router_noBypass = []

HL3_SA_num_routers_noBypass = []
HL3_SA_selected_router_noBypass = []

HL3_co_num_routers_noBypass = []
HL3_co_selected_router_noBypass = []

HL2_SA_num_routers_noBypass = []
HL2_SA_selected_router_noBypass = []

routers_tracker_array_noBypass = np.zeros(shape = (len(ref_routers_cap), len(year)), dtype = np.int64)

for hl4_sa in hl_dict['HL4']['standalone']:
    ref_traffic = node_capacity_profile_array_noBypass_HL4[ref_year - 1, hl4_sa]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL4_SA_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL4[:, hl4_sa] // closest_bigger_router_cap).astype(int) + 1
    HL4_SA_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL4[:, hl4_sa] // closest_bigger_router_cap + 1)

for hl4_co in hl_dict['HL4']['colocated']:
    ref_traffic = node_capacity_profile_array_noBypass_HL4[ref_year - 1, hl4_co]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL4_co_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL4[:, hl4_co] // closest_bigger_router_cap).astype(int) + 1
    HL4_co_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL4[:, hl4_co] // closest_bigger_router_cap + 1)

for hl3_sa in hl_dict['HL3']['standalone']:
    ref_traffic = node_capacity_profile_array_noBypass_HL3[ref_year - 1, hl3_sa]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL3_SA_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL3[:, hl3_sa] // closest_bigger_router_cap).astype(int) + 1
    HL3_SA_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL3[:, hl3_sa] // closest_bigger_router_cap + 1)

for hl3_co in hl_dict['HL3']['colocated']:
    ref_traffic = node_capacity_profile_array_noBypass_HL3[ref_year - 1, hl3_co]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL3_co_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL3[:, hl3_co] // closest_bigger_router_cap).astype(int) + 1
    HL3_co_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL3[:, hl3_co] // closest_bigger_router_cap + 1)

for hl2_sa in hl_dict['HL2']['standalone']:
    ref_traffic = node_capacity_profile_array_noBypass_HL2[ref_year - 1, hl2_sa]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL2_SA_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL2[:, hl2_sa] // closest_bigger_router_cap).astype(int) + 1
    HL2_SA_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL2[:, hl2_sa] // closest_bigger_router_cap + 1)
    
num_routers_HL4_SA_total_yearly_noBypass = np.sum(HL4_SA_num_routers_noBypass, axis = 0)
num_routers_HL4_co_total_yearly_noBypass = np.sum(HL4_co_num_routers_noBypass, axis = 0)
num_routers_HL3_SA_total_yearly_noBypass = np.sum(HL3_SA_num_routers_noBypass, axis = 0)
num_routers_HL3_co_total_yearly_noBypass = np.sum(HL3_co_num_routers_noBypass, axis = 0)
num_routers_HL2_SA_total_yearly_noBypass = np.sum(HL2_SA_num_routers_noBypass, axis = 0)

total_num_router_yearly_noBypass = num_routers_HL4_SA_total_yearly_noBypass + num_routers_HL4_co_total_yearly_noBypass + \
    + num_routers_HL3_SA_total_yearly_noBypass + num_routers_HL3_co_total_yearly_noBypass \
        + num_routers_HL2_SA_total_yearly_noBypass

In [22]:
np.savez(results_dir / "TEFNET24_routers_tracker_array_noBypass.npz", routers_tracker_array = routers_tracker_array_noBypass)

In [23]:
# NoBypass Scenario
HL4_SA_cost_noBypass = []
HL4_co_cost_noBypass = []
HL3_SA_cost_noBypass = []
HL3_co_cost_noBypass = []
HL2_SA_cost_noBypass = []

HL4_SA_power_noBypass = []
HL4_co_power_noBypass = []
HL3_SA_power_noBypass = []
HL3_co_power_noBypass = []
HL2_SA_power_noBypass = []

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL4_SA_selected_router_noBypass]
HL4_SA_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL4_SA_noBypass_power = ref_routers_power_consumption[indices]
HL4_SA_cost_noBypass = [ai * bi for ai, bi in zip(HL4_SA_num_routers_noBypass, HL4_SA_noBypass_cost_metrics)]
HL4_SA_power_noBypass = [ai * bi for ai, bi in zip(HL4_SA_num_routers_noBypass, HL4_SA_noBypass_power)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL4_co_selected_router_noBypass]
HL4_co_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL4_co_noBypass_power = ref_routers_power_consumption[indices]
HL4_co_cost_noBypass = [ai * bi for ai, bi in zip(HL4_co_num_routers_noBypass, HL4_co_noBypass_cost_metrics)]
HL4_co_power_noBypass = [ai * bi for ai, bi in zip(HL4_co_num_routers_noBypass, HL4_co_noBypass_power)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL3_SA_selected_router_noBypass]
HL3_SA_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL3_SA_noBypass_power = ref_routers_power_consumption[indices]
HL3_SA_cost_noBypass = [ai * bi for ai, bi in zip(HL3_SA_num_routers_noBypass, HL3_SA_noBypass_cost_metrics)]
HL3_SA_power_noBypass = [ai * bi for ai, bi in zip(HL3_SA_num_routers_noBypass, HL3_SA_noBypass_power)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL3_co_selected_router_noBypass]
HL3_co_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL3_co_noBypass_power = ref_routers_power_consumption[indices]
HL3_co_cost_noBypass = [ai * bi for ai, bi in zip(HL3_co_num_routers_noBypass, HL3_co_noBypass_cost_metrics)]
HL3_co_power_noBypass = [ai * bi for ai, bi in zip(HL3_co_num_routers_noBypass, HL3_co_noBypass_power)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL2_SA_selected_router_noBypass]
HL2_SA_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL2_SA_noBypass_power = ref_routers_power_consumption[indices]
HL2_SA_cost_noBypass = [ai * bi for ai, bi in zip(HL2_SA_num_routers_noBypass, HL2_SA_noBypass_cost_metrics)]
HL2_SA_power_noBypass = [ai * bi for ai, bi in zip(HL2_SA_num_routers_noBypass, HL2_SA_noBypass_power)]


IP_total_cost_noBypass = np.sum(HL4_SA_cost_noBypass, axis = 0) + np.sum(HL4_co_cost_noBypass, axis = 0) + \
    np.sum(HL3_SA_cost_noBypass, axis = 0) + np.sum(HL3_co_cost_noBypass, axis = 0) \
        + np.sum(HL2_SA_cost_noBypass, axis = 0)
            
IP_total_power_noBypass = np.sum(HL4_SA_power_noBypass, axis = 0) + np.sum(HL4_co_power_noBypass, axis = 0) + \
    np.sum(HL3_SA_power_noBypass, axis = 0) + np.sum(HL3_co_power_noBypass, axis = 0) \
        + np.sum(HL2_SA_power_noBypass, axis = 0)

In [24]:
OTCO_annual = (cost_df_C_IRU_0v5['OPEX'] + cost_df_C_IRU_0v5['CAPEX']).values
TCO_Final_year = (IP_total_cost_noBypass[-1] + OTCO_annual.sum())
TCO_Final_year

np.float64(469975.20264537097)

In [25]:
OTCO_annual = (cost_df_C_IRU_0v01['OPEX'] + cost_df_C_IRU_0v01['CAPEX']).values
TCO_Final_year = (IP_total_cost_noBypass[-1] + OTCO_annual.sum())
TCO_Final_year

np.float64(37855.86464537071)

In [26]:
OTCO_annual = (cost_df_C_IRU_5['OPEX'] + cost_df_C_IRU_5['CAPEX']).values
TCO_Final_year = (IP_total_cost_noBypass[-1] + OTCO_annual.sum())
TCO_Final_year

np.float64(4438418.102645373)

In [27]:
# np.savez(results_dir / "MAN157_IP_Cost_Power_noBypass.npz", 
#          IP_Cost_Cum = IP_total_cost_noBypass, 
#          IP_Power_Cum = IP_total_power_noBypass 
#          )

In [28]:
power_unit_ROADM = 72 #910 # Watt
power_unit_MCS = 10 # Watt
typ_power_400G_BVT = 160 # Watt

In [29]:
Total_BVT_noBypass = analysing.bvt_data['HL4']['HL_BVTNum_All'].sum(axis = 1) + analysing.bvt_data['HL3']['HL_BVTNum_All'].sum(axis = 1)

In [30]:
degree_links_cum = analysing.link_data['HL3']['HL_CDegree_Domain'].sum(axis = 1) +\
    analysing.link_data['HL3']['HL_SuperCDegree_Domain'].sum(axis = 1) +\
    analysing.link_data['HL3']['HL_LDegree_Domain'].sum(axis = 1)

In [31]:
np.diff(Total_BVT_noBypass, prepend=0)

array([3854.,  198.,  210.,  280.,  284.,  374.,  430., 1524., 2386.,
        740.])

In [32]:
# Energy Consumption Between scenarios
energy_consumption_annual_noBypass = (np.diff(Total_BVT_noBypass, prepend=0) * typ_power_400G_BVT +\
    np.diff(degree_links_cum, prepend=0) * power_unit_ROADM +\
        np.diff(Total_BVT_noBypass, prepend=0) / 16 * power_unit_MCS) * 365 * 24 / 1e6


total_served_traffic_noBypass = node_capacity_profile_array_noBypass_HL4.sum(axis = 1)
IP_energy_consumption_annual_noBypass = np.diff(IP_total_power_noBypass * 365 * 24 / 1e6, prepend=0)

# np.savez(results_dir / 'TEFNET24_energy_optical_electrical_noBypass.npz',
#          optical_energy_annual = energy_consumption_annual_noBypass,
#          IP_energy_annual = IP_energy_consumption_annual_noBypass, 
#          total_served_traffic = total_served_traffic_noBypass)

ENERGY_Final_Year = (energy_consumption_annual_noBypass + IP_energy_consumption_annual_noBypass).cumsum()[-1]
ENERGY_Final_Year

np.float64(48743.41692)

In [33]:
analysing.calc_E2E_latency_Total(planner.path_latency_storage, 
								planner.destinations_storage,
    							processing_level_list = processing_level_list, 
                       			save_flag = 0)

array([1347.5, 1382. , 2582.5, ..., 1373. , 2308.5, 2404.5], shape=(2069,))